# Healthcare Provider FWA Risk Scoring — EDA & Modeling Walkthrough

**Project:** Healthcare Provider FWA Risk Scoring & GenAI Review System  
**Data:** Public Kaggle *Healthcare Provider Fraud Detection Analysis* (Train split)  
**Granularity:** Provider-level (5,410 providers, 27 engineered features)

This notebook walks through the same pipeline that lives in `src/`:
1. Load the provider modeling table
2. EDA on fraud distribution, reimbursement, utilization
3. Inspect engineered features and correlations
4. Read model metrics (ROC-AUC, PR-AUC, Brier, log-loss, 5-fold CV)
5. Top risk factors
6. Sample RAG-style provider review

> **Reproducibility:** run `make real-pipeline` (or each script in `src/` individually) to regenerate every artifact this notebook reads. The notebook is read-only on the saved outputs — it does not retrain models.

> **Data disclaimer:** the Kaggle dataset is a public educational resource. It is *not* Manulife / John Hancock data, *not* Long Term Care-specific, and contains no PHI. Synthetic policy text is used only for the RAG demo.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.getcwd()))
import config

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
print('BASE_DIR =', config.BASE_DIR)

## 1. Load the provider modeling table

Built by `src/provider_feature_engineering.py`, which joins three Kaggle CSVs (inpatient, outpatient, beneficiary) and aggregates to one row per `Provider`.

In [ ]:
table_path = os.path.join(config.DATA_PROCESSED, 'provider_modeling_table.csv')
if not os.path.exists(table_path):
    raise FileNotFoundError(
        f'{table_path} not found. Place Kaggle CSVs in data/raw/ and run '
        f'`python src/provider_feature_engineering.py` first.'
    )
df = pd.read_csv(table_path)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
fraud_rate = df['PotentialFraud'].mean()
print(f'Providers analyzed:   {len(df):>6,}')
print(f'Fraudulent providers: {df["PotentialFraud"].sum():>6,}')
print(f'Fraud rate:           {fraud_rate:>6.2%}')
print()
print('Missing values per column:')
print(df.isnull().sum().sort_values(ascending=False).head(10))

## 2. EDA — fraud signal in headline features

Inspect the three signals that the model later ranks highest: `max_admission_duration`, `total_reimbursed`, and `inpatient_claim_count`. Each is plotted by fraud label so the separability is visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['max_admission_duration', 'total_reimbursed', 'inpatient_claim_count']):
    legit = df[df['PotentialFraud'] == 0][col].dropna()
    fraud = df[df['PotentialFraud'] == 1][col].dropna()
    # log-scale for skewed dollar/count features
    if col in ('total_reimbursed', 'inpatient_claim_count'):
        legit = np.log1p(legit); fraud = np.log1p(fraud)
        ax.set_xlabel(f'log1p({col})')
    else:
        ax.set_xlabel(col)
    ax.hist(legit, bins=40, alpha=0.55, color='#4CAF50', label='Legit', density=True)
    ax.hist(fraud, bins=40, alpha=0.55, color='#F44336', label='Fraud', density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Fraud rate by provider-volume quintile — sanity check that volume alone is
# not the entire signal (i.e. risk should be elevated but distributed).
df['volume_bucket'] = pd.qcut(df['total_claims'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
vb = df.groupby('volume_bucket')['PotentialFraud'].agg(['mean', 'count']).round(3)
vb.columns = ['fraud_rate', 'n_providers']
vb

## 3. Feature correlation

Correlation matrix of the top numeric features. Look for clusters of redundant features (e.g. `total_reimbursed` vs `total_inpatient_reimbursed`) and for features that correlate weakly with each other but strongly with the label — those are the most informative.

In [ ]:
key_cols = [
    'max_admission_duration', 'total_reimbursed', 'total_deductible_amount',
    'total_inpatient_reimbursed', 'inpatient_claim_count', 'inpatient_ratio',
    'unique_beneficiaries', 'avg_chronic_conditions', 'PotentialFraud',
]
key_cols = [c for c in key_cols if c in df.columns]
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(df[key_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature correlation — provider-level table')
plt.tight_layout(); plt.show()

## 4. Model metrics

Reads `outputs/reports/model_metrics.json` produced by `src/modeling.py`. Each model carries both held-out test metrics (ROC-AUC, PR-AUC, F1, Brier, log-loss) and 5-fold cross-validated AUC (mean ± std).

In [ ]:
metrics_path = os.path.join(config.OUTPUTS_REPORTS, 'model_metrics.json')
if not os.path.exists(metrics_path):
    print('Run `python src/modeling.py` first.')
else:
    with open(metrics_path) as f:
        m = json.load(f)
    print('Data source:', m.get('_data_source'))
    print('Evaluation:', m.get('_evaluation'))
    model_rows = {k: v for k, v in m.items() if not k.startswith('_')}
    metrics_df = pd.DataFrame(model_rows).T
    # display test-set + CV columns if present
    display_cols = [c for c in ['roc_auc', 'pr_auc', 'precision', 'recall', 'f1',
                                'brier', 'log_loss', 'cv_auc_mean', 'cv_auc_std']
                    if c in metrics_df.columns]
    metrics_df = metrics_df[display_cols].sort_values('pr_auc' if 'pr_auc' in display_cols else 'roc_auc',
                                                       ascending=False)
    metrics_df

In [ ]:
from IPython.display import Image, display
for fig_name in ['roc_curve.png', 'precision_recall_curve.png',
                 'calibration_curve.png', 'feature_importance.png']:
    path = os.path.join(config.OUTPUTS_FIGURES, fig_name)
    if os.path.exists(path):
        print(fig_name); display(Image(path, width=650))

## 5. Top risk factors

Generated by `src/explainability.py` — feature importances from the best model, ranked.

In [ ]:
rf_path = os.path.join(config.OUTPUTS_REPORTS, 'top_risk_factors.csv')
if os.path.exists(rf_path):
    pd.read_csv(rf_path).head(10)
else:
    print('Run `python src/explainability.py` first.')

## 6. Sample RAG-style provider review

Generated by `src/rag_claim_review.py`. The review combines model risk indicators with retrieved policy/audit evidence (TF-IDF over `data/documents/policy_rules.txt`) into an auditable, human-readable packet.

In [ ]:
reviews_dir = config.OUTPUTS_REVIEWS
files = sorted([f for f in os.listdir(reviews_dir) if f.endswith('.txt')]) if os.path.exists(reviews_dir) else []
print(f'Reviews available: {len(files)}')
if files:
    sample = os.path.join(reviews_dir, files[0])
    print(f'\n--- {files[0]} ---')
    print(open(sample).read())

## 7. Summary

Provider-level FWA workflow on a real public reference dataset:

- **Data:** 558K+ inpatient/outpatient claims joined with beneficiary records → 5,410 providers × 27 features.
- **Models:** Logistic Regression, Random Forest, Gradient Boosting; Isolation Forest as unsupervised baseline.
- **Evaluation:** held-out 80/20 test + 5-fold stratified cross-validation; ROC-AUC, PR-AUC, F1, Brier score, log-loss.
- **Explainability:** model-derived feature importances + business-readable per-provider explanations.
- **RAG layer:** TF-IDF retrieval over audit policy text → structured review packets for analyst sign-off.

All artifacts are reproducible from `make real-pipeline`.